# Spatial transcriptomics of Crohn's disease: cell segmentation errors in Xenium data

Companion notebook for the TensorOmics note *Spatial transcriptomics of Crohn's disease: cell segmentation errors in Xenium data*. It runs the note's code blocks in order.

**Data.** Download `xenium5k_scVI_CellCharter_batch2_reannot_SCP.h5ad` (about 9 GB) from the
[Broad Single Cell Portal, study SCP3259](https://singlecell.broadinstitute.org/single_cell/study/SCP3259)
(free, Google sign-in) and put it in `data/`. See `data/DATA_SOURCES.md`.

The note: https://tensoromics.com/notes/spatial-transcriptomics-crohns-cell-overlap/

## How close are the neighboring cells?

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# open the file; we read only the cell table (obs), the 9 GB matrix stays on disk
f = h5py.File("data/xenium5k_scVI_CellCharter_batch2_reannot_SCP.h5ad", "r")
obs = f["obs"]

def column(name):
    g = obs[name]
    # numbers are stored as plain arrays: read them as is
    if isinstance(g, h5py.Dataset):
        return g[:]
    # labels are stored as codes plus a list of categories: turn codes into names
    return g["categories"].asstr()[:][g["codes"][:]]

# short names (left) for the columns we need (right), one row of `cells` per cell;
# dots is n_counts, the cell's RNA molecules summed over all genes in the count matrix
# (transcript_counts is a little higher, because it also counts molecules not in the shared matrix)
cols = {"section": "biosample_id", "x": "x_centroid", "y": "y_centroid",
        "area": "cell_area", "dots": "n_counts",
        "cell_type": "megagut_lvl1_majority_voting", "fine_type": "megagut_lvl3_majority_voting",
        "confidence": "megagut_lvl1_conf_score", "controls": "control_probe_counts"}
cells = pd.DataFrame({short: column(name) for short, name in cols.items()})

# number of cells (rows) and columns
print(cells.shape)

In [ ]:
from scipy.spatial import cKDTree

# the cell centers, as an array with one row per cell and two columns (x, y)
xy = cells[["x", "y"]].to_numpy()

# for each cell: distance to its nearest cell, and that cell's row number
nn_dist = np.zeros(len(cells))
nn = np.zeros(len(cells), dtype=int)

# `rows` holds the row numbers of the cells in one section
for section, rows in cells.groupby("section").indices.items():
    # build the tree from this section's centers, then ask for the 2 closest centers to each cell;
    # the closest is the cell itself (distance 0), the second is its nearest neighbor
    dist, j = cKDTree(xy[rows]).query(xy[rows], k=2)
    nn_dist[rows] = dist[:, 1]
    # j counts within the section; rows[j] turns it back into a row number of `cells`
    nn[rows] = rows[j[:, 1]]
cells["nn_dist"] = nn_dist
cells["nn"] = nn

In [ ]:
# radius of a circle with the same area as the cell
r = np.sqrt(cells.area.to_numpy() / np.pi)
cells["diameter"] = 2 * r

# a cell overlaps its nearest neighbor when the two circles overlap;
# r is the cell's own radius, and r[nn] is the radius of its nearest neighbor
cells["overlaps"] = cells.nn_dist < r + r[nn]

# the median is the middle value: half the cells are below it, half above
print(f"median distance to the nearest cell: {cells.nn_dist.median():.1f} µm")
print(f"median cell diameter: {cells.diameter.median():.1f} µm")
print(f"cells that overlap their nearest neighbor: {cells.overlaps.mean():.0%}")

# the same share, section by section (the lowest and the highest of the 16)
by_section = cells.groupby("section").overlaps.mean()
print(f"per section: {by_section.min():.0%} to {by_section.max():.0%}")

# how much the share depends on the circle size: every radius 10% smaller, then 10% larger
for scale in [0.9, 1.1]:
    print(f"radii x {scale}: {(cells.nn_dist < scale * (r + r[nn])).mean():.0%} overlap")

### What the packing looks like

In [ ]:
from matplotlib.patches import Circle

# one color per cell type, most common type first (the same colors as in part 1)
colors = dict(zip(cells.cell_type.value_counts().index, plt.get_cmap("tab10").colors))
one = cells[cells.section == "0043587_affected"]

# the typical square: cut the section into 50 µm squares, and find the share of overlapping cells
# in each; among squares with at least 30 cells, take the one closest to the share for all cells
w = 50
grid = one.groupby([one.x // w, one.y // w]).overlaps.agg(["mean", "size"])
grid = grid[grid["size"] >= 30]
gx, gy = (grid["mean"] - cells.overlaps.mean()).abs().idxmin()
squares = {"inside the ulcer patch from part 1": (4409.39, 837.0),
           "a typical square": (gx * w, gy * w)}

fig, axes = plt.subplots(1, 2, figsize=(11, 5.6), layout="constrained")
shown = set()
for ax, (name, (x0, y0)) in zip(axes, squares.items()):
    # draw the cells near the square too, so that circles at the edge are complete
    around = one[(one.x > x0 - 10) & (one.x < x0 + w + 10) & (one.y > y0 - 10) & (one.y < y0 + w + 10)]
    for i, cell in around.iterrows():
        ax.add_patch(Circle((cell.x, cell.y), r[i], facecolor=colors[cell.cell_type], alpha=0.5,
                            edgecolor="black", lw=0.5))
    # a line from each cell in the square to its nearest cell: red if the circles overlap
    square = around[(around.x >= x0) & (around.x < x0 + w) & (around.y >= y0) & (around.y < y0 + w)]
    shown |= set(square.cell_type)
    for i, cell in square.iterrows():
        ax.plot([cell.x, xy[nn[i], 0]], [cell.y, xy[nn[i], 1]], lw=1.4,
                color="red" if cell.overlaps else "gray")
    # a 10 µm scale bar in the bottom-left corner
    ax.plot([x0 + 3, x0 + 13], [y0 + w - 3, y0 + w - 3], color="black", lw=3)
    ax.text(x0 + 8, y0 + w - 4, "10 µm", ha="center", va="bottom", backgroundcolor="white")
    ax.set_xlim(x0, x0 + w)
    ax.set_ylim(y0 + w, y0)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{name}: {square.overlaps.mean():.0%} overlap")
    print(f"{name}: {len(square)} cells, {square.overlaps.mean():.0%} overlap their nearest cell")

# one legend for both squares: the cell types drawn, most common first
shown = [t for t in colors if t in shown]
handles = [plt.Line2D([0], [0], marker="o", ls="", ms=9, color=colors[t], alpha=0.6) for t in shown]
fig.legend(handles, shown, loc="outside lower center", ncol=4, frameon=False)
plt.show()

### Checking the circles against real outlines

## Are neighbors the same cell type?

In [ ]:
# each cell's broad and finer type, and whether it overlaps its nearest neighbor (True/False)
types = cells.cell_type.to_numpy()
fine = cells.fine_type.to_numpy()
over = cells.overlaps.to_numpy()

# baseline: shuffle the labels within each section, so the cell types keep their numbers
# but are placed at random
rng = np.random.default_rng(0)
def shuffle(labels):
    out = labels.copy()
    for section, rows in cells.groupby("section").indices.items():
        out[rows] = rng.permutation(labels[rows])
    return out

# among cells that overlap their nearest neighbor: how often is the neighbor a different type?
for name, labels in [("7 broad types", types), ("54 finer types", fine)]:
    # labels[nn] is the type of each cell's nearest neighbor; [over] keeps the overlapping cells
    differ = labels[over] != labels[nn][over]
    s = shuffle(labels)
    differ_random = s[over] != s[nn][over]
    print(f"{name}: different type {differ.mean():.0%}, if placed at random {differ_random.mean():.0%}")

In [ ]:
# for each broad type: the share of its overlapping neighbors that are a different type
differ = pd.Series(types[over] != types[nn][over], index=types[over])
by_type = differ.groupby(level=0).mean().sort_values()
print(by_type.round(2))

# the same share with the types placed at random (shuffled within each section)
s = shuffle(types)
by_type_random = pd.Series(s[over] != s[nn][over], index=s[over]).groupby(level=0).mean()

# the most common pairs of different types, as a share of all such pairs
# a: the type of each overlapping cell, b: the type of its neighbor; we keep pairs where they differ
# and sort each pair by name, so "Myeloid + T" and "T + Myeloid" count as the same pair
a, b = types[over], types[nn][over]
pairs = pd.Series([" + ".join(sorted(p)) for p in zip(a[a != b], b[a != b])])
top = pairs.value_counts(normalize=True).head(5)
print(top.round(2))

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")

# left: one bar per type; the black tick is the share if the types were placed at random
left.barh(by_type.index, by_type, color="C0", label="observed")
left.scatter(by_type_random[by_type.index], by_type.index, color="black", marker="|", s=300,
             label="if placed at random")
left.set_xlim(0, 1)
left.set_xlabel("share of touching neighbors that are a different type")
left.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)

# right: the five most common pairs, the most common at the top
right.barh(top.index[::-1], top[::-1], color="gray")
right.set_xlabel("share of all touching pairs of different types")
plt.show()

## Do cells hold genes from the cells next to them?

In [ ]:
markers = ["EPCAM", "CD3E", "MZB1", "CLDN5", "SOX10", "CD163", "PDGFRA"]
genes = f["var"]["_index"].asstr()[:]

# lookup[column] is 0-6 for the seven marker genes (their place in `markers`), and -1 for all others
lookup = np.full(len(genes), -1)
for k, g in enumerate(markers):
    lookup[np.flatnonzero(genes == g)[0]] = k

# the matrix is stored row by row (CSR): for each nonzero entry, its gene (indices) and its
# count (data); indptr says where each cell's entries start
X = f["layers"]["counts"]
indptr = X["indptr"][:]
# the result: one row per cell, one column per marker gene
counts = np.zeros((len(cells), len(markers)), dtype=np.int32)

# read 50 million entries at a time, so the chunk fits in memory
step = 50_000_000
for start in range(0, indptr[-1], step):
    # the gene (column number) of each entry in this chunk
    idx = X["indices"][start:start + step]
    # the entries that belong to a marker gene
    keep = np.flatnonzero(lookup[idx] >= 0)
    # the cell of each kept entry: the last cell whose entries begin at or before it
    cell = np.searchsorted(indptr, start + keep, side="right") - 1
    # copy the counts into the cell's row, in the marker gene's column
    counts[cell, lookup[idx[keep]]] = X["data"][start:start + step][keep]

# total dots of each marker gene, over all cells
print(pd.Series(counts.sum(axis=0), index=markers))

In [ ]:
def distance_to(kind):
    # distance from each cell's center to the nearest cell of one type, in the same section
    d = np.zeros(len(cells))
    is_kind = (cells.cell_type == kind).to_numpy()
    for section, rows in cells.groupby("section").indices.items():
        # the tree holds this section's cells of that type; we ask it about all cells in the section
        source = rows[is_kind[rows]]
        dist, _ = cKDTree(xy[source]).query(xy[rows], k=2)
        # a cell of that type finds itself first, so it takes the second distance
        d[rows] = np.where(is_kind[rows], dist[:, 1], dist[:, 0])
    return d

cells["to_epithelial"] = distance_to("Epithelial")

In [ ]:
# the distance groups, in µm
bins = [0, 5, 10, 15, 20, 30, 50, 100, np.inf]
bin_names = ["0-5", "5-10", "10-15", "15-20", "20-30", "30-50", "50-100", "over 100"]
dots = cells.dots.to_numpy()

def by_distance(values, distance, keep):
    # dots of one gene (or probe) per 1,000 dots, that is, per 1,000 RNA molecules,
    # for the cells in `keep`, by distance (µm);
    # pd.cut puts each cell into its distance group
    t = pd.DataFrame({"µm": pd.cut(distance[keep], bins, labels=bin_names),
                      "n": values[keep], "dots": dots[keep]})
    g = t.groupby("µm", observed=True)
    return 1000 * g.n.sum() / g.dots.sum()

# each cell's EPCAM dots, whether it is epithelial, and its distance to the nearest epithelial cell
epcam = counts[:, markers.index("EPCAM")]
epi = (cells.cell_type == "Epithelial").to_numpy()
near = cells.to_epithelial.to_numpy()

# the non-epithelial cells (~epi means "not epithelial"), grouped by distance
t = pd.DataFrame({"µm": pd.cut(near[~epi], bins, labels=bin_names),
                  "epcam": epcam[~epi], "dots": dots[~epi]})
g = t.groupby("µm", observed=True)
table = pd.DataFrame({"cells": g.size(),
                      "EPCAM per 1,000 molecules": 1000 * g.epcam.sum() / g.dots.sum(),
                      "share with an EPCAM dot": g.epcam.apply(lambda x: (x > 0).mean())})
print(table.round(3))
# for comparison: the epithelial cells' own EPCAM level
print(f"epithelial cells themselves: {1000 * epcam[epi].sum() / dots[epi].sum():.1f} EPCAM per 1,000 molecules")

### Three other explanations

In [ ]:
# MZB1 dots; cells whose label is at least 99% sure; and cells that are not B or plasma cells
mzb1 = counts[:, markers.index("MZB1")]
confident = cells.confidence.to_numpy() >= 0.99
not_b = (cells.cell_type != "B and B plasma").to_numpy()

# one column per check; & means "and", so ~epi & confident keeps confident non-epithelial cells
checks = pd.DataFrame({
    # 1. only cells whose label the classifier is at least 99% sure of
    "EPCAM, confident labels": by_distance(epcam, near, ~epi & confident),
    # 2. negative-control probes, which match no gene
    "control probes": by_distance(cells.controls.to_numpy(), near, ~epi),
    # 3. a plasma-cell gene, in cells that are neither epithelial nor B/plasma
    "MZB1": by_distance(mzb1, near, ~epi & not_b),
})
print(checks.round(3))

In [ ]:
# EPCAM in all non-epithelial cells (from the table above), next to the three checks
lines = pd.concat([table["EPCAM per 1,000 molecules"].rename("EPCAM, all labels"), checks], axis=1)

# each gene in the color of the cell type that makes it, as in Figures 4 and 9
epi_color, b_color = colors["Epithelial"], colors["B and B plasma"]
fig, ax = plt.subplots(figsize=(7, 4))
for name, color, style in [("EPCAM, all labels", epi_color, "-"), ("EPCAM, confident labels", epi_color, "--"),
                           ("control probes", "gray", "-"), ("MZB1", b_color, "-")]:
    # divide by the level far away (over 100 µm, the last group), so every line ends at 1
    level = lines[name] / lines[name].iloc[-1]
    # the legend gives each line's rise at 0-5 µm
    ax.plot(range(len(level)), level, color=color, ls=style, marker="o",
            label=f"{name}: {level.iloc[0]:.1f}× at 0-5 µm")
ax.axhline(1, color="black", lw=0.8)
ax.set_xticks(range(len(bin_names)), bin_names)
ax.set_yscale("log")
ax.set_yticks([0.5, 1, 2, 5, 10, 20], ["0.5", "1", "2", "5", "10", "20"])
ax.set_xlabel("distance to the nearest epithelial cell (µm)")
ax.set_ylabel("per 1,000 RNA molecules, relative to far away")
ax.legend(frameon=False, fontsize=9)
plt.show()

### One marker gene for each cell type

In [ ]:
# each marker gene, with the cell type that makes it (in the order of `markers`)
sources = {"EPCAM": "Epithelial", "CD3E": "T and NK cells", "MZB1": "B and B plasma",
           "CLDN5": "Endothelial", "SOX10": "Neural", "CD163": "Myeloid", "PDGFRA": "Mesenchymal"}
source_types = list(sources.values())

# A: the share of each gene's RNA molecules in each cell type, in %
# (sum each gene's dots over the cells of each type, then divide by the gene's total)
per_type = pd.DataFrame(counts, columns=markers).groupby(cells.cell_type.to_numpy()).sum()
share = (100 * per_type / per_type.sum()).T.loc[markers, source_types]

# B: each gene in the cells of all other types, by distance to the nearest cell of its own type
rise = {}
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6), layout="constrained",
                                  gridspec_kw=dict(width_ratios=[1, 1.35]))
for gene, kind in sources.items():
    other = (cells.cell_type != kind).to_numpy()
    level = by_distance(counts[:, markers.index(gene)], distance_to(kind), other)
    # divide by the level far away (over 100 µm, the last group), so every line ends at 1
    level = level / level.iloc[-1]
    rise[gene] = level.iloc[0]
    # the same color for each type as in Figure 4
    right.plot(range(len(level)), level, marker="o", color=colors[kind],
               label=f"{gene}, from {kind}: {level.iloc[0]:.0f}× at 0-5 µm")
print(pd.DataFrame({"made by": source_types, "% of its RNA in that type": np.diag(share),
                    "rise at 0-5 µm": list(rise.values())}, index=markers).round(1))

# A: the shares as a grid; the diagonal is each gene in its own type
left.imshow(share, cmap="Blues", vmin=0, vmax=100)
for i in range(len(markers)):
    for j in range(len(source_types)):
        left.text(j, i, f"{share.iloc[i, j]:.0f}", ha="center", va="center", fontsize=9,
                  color="white" if share.iloc[i, j] > 50 else "black")
left.set_xticks(range(len(source_types)), source_types, rotation=40, ha="right")
left.set_yticks(range(len(markers)), markers)
left.set_title("A · share of each gene's RNA, by cell type (%)", loc="left")

right.axhline(1, color="black", lw=0.8)
right.set_xticks(range(len(bin_names)), bin_names)
right.set_yscale("log")
right.set_yticks([1, 2, 5, 10, 20, 40], ["1", "2", "5", "10", "20", "40"])
right.set_xlabel("distance to the nearest cell that makes the gene (µm)")
right.set_ylabel("per 1,000 RNA molecules, relative to far away")
right.set_title("B · in the cells of other types, by distance", loc="left")
right.legend(frameon=False, fontsize=9)
plt.show()

## Does it change what we learn about T cells?

In [ ]:
# the two groups: T cells within 10 µm of an epithelial cell (0), and more than 50 µm away (1)
is_t = (cells.cell_type == "T and NK cells").to_numpy()
group = np.full(len(cells), -1)
group[is_t & (near < 10)] = 0
group[is_t & (near > 50)] = 1

# T cells in each group, section by section; keep sections with at least 300 in each group
n = pd.crosstab(cells.section[group >= 0], group[group >= 0])
use = n[(n[0] >= 300) & (n[1] >= 300)].index
print(f"{len(use)} of {len(n)} sections: {n.loc[use, 0].sum():,} near and {n.loc[use, 1].sum():,} far T cells")

# a slot for each section and group: the near T cells of section k go to slot 2k, the far ones
# to slot 2k + 1; all other cells get -1
position = pd.Series(np.arange(len(use)), index=use)
slot = np.full(len(cells), -1)
kept = (group >= 0) & cells.section.isin(use).to_numpy()
slot[kept] = 2 * position[cells.section[kept]].to_numpy() + group[kept]

# the dots of every gene, added up in each slot, in the epithelial cells, and in all cells;
# we read the matrix in chunks, as before
by_slot = np.zeros((2 * len(use), len(genes)))
epi_total = np.zeros(len(genes))
all_total = np.zeros(len(genes))
for start in range(0, indptr[-1], step):
    idx = X["indices"][start:start + step]
    data = X["data"][start:start + step]
    # the cell, and the slot, of every entry in this chunk
    cell = np.searchsorted(indptr, start + np.arange(len(idx)), side="right") - 1
    sl = slot[cell]
    ok = sl >= 0
    # bincount adds up the counts for each (slot, gene) pair
    by_slot += np.bincount(sl[ok] * len(genes) + idx[ok], weights=data[ok],
                           minlength=by_slot.size).reshape(by_slot.shape)
    is_epi = epi[cell]
    epi_total += np.bincount(idx[is_epi], weights=data[is_epi], minlength=len(genes))
    all_total += np.bincount(idx, weights=data, minlength=len(genes))

# each gene per 1,000 RNA molecules in each slot; then, section by section, near over far
# (0.01 is added to both, so that a gene absent far away doesn't divide by zero)
level = 1000 * by_slot / by_slot.sum(axis=1, keepdims=True)
fold = (level[0::2] + 0.01) / (level[1::2] + 0.01)

# the median over the sections: half of the sections are above it, half below
de = pd.DataFrame({"times higher": np.median(fold, axis=0),
                   "sections higher": (fold > 1).sum(axis=0),
                   "% in epithelial cells": 100 * epi_total / all_total,
                   "dots near": by_slot[0::2].sum(axis=0).astype(int)}, index=genes)
# keep genes with at least 100 dots in the near T cells, and rank them, the highest first
de = de[de["dots near"] >= 100].sort_values("times higher", ascending=False)
de["rank"] = np.arange(1, len(de) + 1)
print(de.head(10).round(1))

# how many of the top genes are epithelial genes (most of their RNA in epithelial cells)
for n_top in [20, 50]:
    print(f"top {n_top}: {(de.head(n_top)['% in epithelial cells'] > 50).sum()} epithelial genes")
# a gene that T cells in the epithelium are known to make
print(de.loc[["ITGAE"]].round(1))

In [ ]:
# the top 20 genes, then ITGAE; for each, its fold change in every section (a column of `fold`)
show = pd.concat([de.head(20), de.loc[["ITGAE"]]])
column = pd.Series(np.arange(len(genes)), index=genes)
per_section = [fold[:, column[g]] for g in show.index]
# epithelial genes in the epithelial color, the T-cell gene in the T-cell color
color = [colors["Epithelial"] if share > 50 else colors["T and NK cells"]
         for share in show["% in epithelial cells"]]
# the rows from the top down, with a gap before ITGAE
y = np.r_[np.arange(20), 21.2]

fig, ax = plt.subplots(figsize=(7, 6.5))
boxes = ax.boxplot(per_section, positions=y, orientation="horizontal", widths=0.6, showfliers=False,
                   patch_artist=True)
for box, c in zip(boxes["boxes"], color):
    box.set(facecolor=c, alpha=0.35)
for median in boxes["medians"]:
    median.set(color="black")
# one dot per section, moved up or down a little at random so that the dots don't hide each other
jitter = np.random.default_rng(1)
for yi, values, c in zip(y, per_section, color):
    ax.scatter(values, yi + jitter.uniform(-0.2, 0.2, len(values)), s=10, color=c, zorder=3)
ax.axvline(1, color="black", lw=0.8)
ax.set_xscale("log")
ax.set_xticks([0.5, 1, 2, 5, 10, 20, 50, 100], ["0.5", "1", "2", "5", "10", "20", "50", "100"])
ax.set_yticks(y, [f"{g} ({r})" for g, r in zip(show.index, show["rank"])])
ax.invert_yaxis()
ax.set_xlabel("times higher in T cells near the epithelium (one dot per section)")
handles = [plt.Rectangle((0, 0), 1, 1, color=colors[t], alpha=0.6) for t in ["Epithelial", "T and NK cells"]]
ax.legend(handles, ["epithelial gene (most of its RNA in epithelial cells)", "T-cell gene"],
          frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
plt.show()

## What others have measured